# Import modules

In [22]:
import os
import sys
import requests                      # HTTP client for API calls
import pandas as pd                  # Tabular data handling
from datetime import datetime        # Datetime handling
from typing import Iterable, Optional, Dict, Union
import matplotlib.pyplot as plt
import yfinance as yf
from pprint import pprint as pp
from IPython.display import display



Import internal modules

In [23]:
from src.fetch_lse_tickers import get_ftse100
from src.exchange_rates_v2 import get_share_prices_2_with_fundamentals
from src.plot_shares_ROI import plot_candles_volatility_volume_roi as ROI
from src.extract_latest_fundamentals import extract_latest_fundamentals
from src.detect_undervalued import detect_undervalued
from src.utils.email_sender import send_email_html_multi_inline_images 
from src.utils.email_undervalued_shares import build_undervalued_shares_email_with_images
from src.purchase_price import get_purchase_price
from src.utils.build_portfolio_email_html import build_roi_email_from_df
from src.utils.weighted_avg import weighted_avg



Ensure repo root is on PYTHONPATH (CI safety)

In [24]:
REPO_ROOT = os.path.abspath(os.getcwd())
SRC_PATH = os.path.join(REPO_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# print(f"REPO_ROOT: {REPO_ROOT}")
# print(f"SRC_PATH: {SRC_PATH}")

Setup Paths for Portfolio, Images and read env variables for Email sender

In [25]:
from pathlib import Path
from src.debug_print import debug_print

# Resolve project root assuming notebook is in finance/notebooks/
PROJECT_ROOT = Path.cwd()#.parent
portfolio_dir = PROJECT_ROOT / "PORTFOLIO" 
os.makedirs(portfolio_dir, exist_ok=True)
portfolio_path = portfolio_dir / "purchases.csv"
# print(f"Project root: {PROJECT_ROOT}")
# print(f"Portfolio path: {portfolio_path}")
    
# Set up output directory for images
pics_dir = os.path.join(os.getcwd(), "output")
os.makedirs(pics_dir, exist_ok=True)

# Load required environment variables
try:
    REQUIRED_ENV_VARS = [
        "EMAIL_USER",
        "EMAIL_SENDER",
        "EMAIL_SENDER_PSW",
    ]

    missing = [v for v in REQUIRED_ENV_VARS if not os.getenv(v)]
    if missing:
        raise RuntimeError(
            f"{debug_print()}\n Missing required environment variables: {', '.join(missing)}"
        )

    email_user = os.getenv("EMAIL_USER")
    email_sender_psw = os.getenv("EMAIL_SENDER_PSW")
    email_sender = os.getenv("EMAIL_SENDER")
except Exception as e:
    print(f"{debug_print()}\n Could not find env vars for email users\n{type(e).__name__}: {e}")


Load purchase orders - buys_out

In [26]:

csv_files = [f for f in os.listdir(portfolio_dir) if f.endswith('.csv') and f != 'purchases.csv']
buys = []

for file in csv_files:
    purchase_path_temp = portfolio_dir / file
    buys_temp = pd.read_csv(purchase_path_temp, parse_dates=['Time'])
    buys_temp = buys_temp.drop(columns=['Unnamed: 0'], errors='ignore')
    buys.append(buys_temp)
buys_out = pd.concat(buys, ignore_index=True)
buys_out = buys_out[buys_out['Ticker'].notnull()]
display(buys_out)
# display(buys_out['No. of shares']*buys_out['Price / share']/buys_out['Exchange rate'] + buys_out['Stamp duty reserve tax'].fillna(0))
# display(buys_out['Total'])

import pandas as pd

portfolio_2 = pd.DataFrame({
    'Action': pd.Series(dtype='string'),
    'Purchase_Date(weighted_avg)': pd.Series(dtype='datetime64[ns]'),
    'No. of shares': pd.Series(dtype='float64'),
    'Purchase_Price(weighted_avg)': pd.Series(dtype='float64'),
    'Tax': pd.Series(dtype='float64'),
    'invested_amount': pd.Series(dtype='float64'),
    'Current_Price': pd.Series(dtype='float64'),
    'Currency': pd.Series(dtype='string'),
    'ROI': pd.Series(dtype='float64'),
    'Target_ROI': pd.Series(dtype='float64'),
    'ROI_reached': pd.Series(dtype='bool'),
    'ID': pd.Series(dtype='object'),  # set{str}
})


,Action,Time,ISIN,Ticker,Name,Notes,ID,No. of shares,Price / share,Currency (Price / share),Exchange rate,Currency (Result),Total,Currency (Total),Stamp duty reserve tax,Currency (Stamp duty reserve tax)
1,Market buy,2026-01-05 08:00:39,GB00B2QPKJ12,FRES,Fresnillo,NaN,EOF44399974076,2.86084,3478.00,GBX,100.0,GBP,100.00,GBP,0.50,GBP
2,Market buy,2026-01-06 08:00:34,JE00B4T3BW64,GLEN,Glencore,NaN,EOF44500414389,11.76609,424.95,GBX,100.0,GBP,50.00,GBP,NaN,NaN
3,Limit buy,2026-01-07 15:01:11,GB00BM8PJY71,NWG,NatWest,NaN,EOF44555166930,10.00000,635.00,GBX,100.0,GBP,63.82,GBP,0.32,GBP
4,Limit buy,2026-01-08 08:21:36,GB0007188757,RIO,Rio Tinto,NaN,EOF44600259487,1.00000,6200.00,GBX,100.0,GBP,62.31,GBP,0.31,GBP
5,Limit buy,2026-01-14 10:43:37,GB00B63H8491,RR,Rolls-Royce,NaN,EOF44900685521,5.00000,1292.00,GBX,100.0,GBP,64.92,GBP,0.32,GBP
6,Limit buy,2026-01-15 08:01:36,GB00B2QPKJ12,FRES,Fresnillo,NaN,EOF44912169636,1.00000,3646.00,GBX,100.0,GBP,36.64,GBP,0.18,GBP


DataFrames for weighted avg test

In [27]:
df_portfolio = buys_out[buys_out['ID']=='EOF44399974076'].copy()
df_input_share = buys_out[buys_out['ID']=='EOF44912169636'].copy()
display(portfolio_2)
display(df_portfolio)
display(df_input_share)


,Action,Purchase_Date(weighted_avg),No. of shares,Purchase_Price(weighted_avg),Tax,invested_amount,Current_Price,Currency,ROI,Target_ROI,ROI_reached,ID


,Action,Time,ISIN,Ticker,Name,Notes,ID,No. of shares,Price / share,Currency (Price / share),Exchange rate,Currency (Result),Total,Currency (Total),Stamp duty reserve tax,Currency (Stamp duty reserve tax)
1,Market buy,2026-01-05 08:00:39,GB00B2QPKJ12,FRES,Fresnillo,NaN,EOF44399974076,2.86084,3478.0,GBX,100.0,GBP,100.0,GBP,0.5,GBP


,Action,Time,ISIN,Ticker,Name,Notes,ID,No. of shares,Price / share,Currency (Price / share),Exchange rate,Currency (Result),Total,Currency (Total),Stamp duty reserve tax,Currency (Stamp duty reserve tax)
6,Limit buy,2026-01-15 08:01:36,GB00B2QPKJ12,FRES,Fresnillo,NaN,EOF44912169636,1.0,3646.0,GBX,100.0,GBP,36.64,GBP,0.18,GBP


In [46]:
idx = buys_out.index
portfolio_test_df = buys_out.loc[:5].copy()
portfolio_test_df['Capital_invested'] = (
    portfolio_test_df['No. of shares']
    * portfolio_test_df['Price / share']
    / portfolio_test_df['Exchange rate']
    + portfolio_test_df['Stamp duty reserve tax'].fillna(0)
).round(2)

input__test_df = buys_out.loc[[6]].copy()  # NOTE: double brackets → DataFrame
input__test_df['Capital_invested'] = (
    input__test_df['No. of shares']
    * input__test_df['Price / share']
    / input__test_df['Exchange rate']
    + input__test_df['Stamp duty reserve tax'].fillna(0)
).round(2)

display(portfolio_test_df)
display(input__test_df)


,Action,Time,ISIN,Ticker,Name,Notes,ID,No. of shares,Price / share,Currency (Price / share),Exchange rate,Currency (Result),Total,Currency (Total),Stamp duty reserve tax,Currency (Stamp duty reserve tax),Capital_invested
1,Market buy,2026-01-05 08:00:39,GB00B2QPKJ12,FRES,Fresnillo,NaN,EOF44399974076,2.86084,3478.00,GBX,100.0,GBP,100.00,GBP,0.50,GBP,100.00
2,Market buy,2026-01-06 08:00:34,JE00B4T3BW64,GLEN,Glencore,NaN,EOF44500414389,11.76609,424.95,GBX,100.0,GBP,50.00,GBP,NaN,NaN,50.00
3,Limit buy,2026-01-07 15:01:11,GB00BM8PJY71,NWG,NatWest,NaN,EOF44555166930,10.00000,635.00,GBX,100.0,GBP,63.82,GBP,0.32,GBP,63.82
4,Limit buy,2026-01-08 08:21:36,GB0007188757,RIO,Rio Tinto,NaN,EOF44600259487,1.00000,6200.00,GBX,100.0,GBP,62.31,GBP,0.31,GBP,62.31
5,Limit buy,2026-01-14 10:43:37,GB00B63H8491,RR,Rolls-Royce,NaN,EOF44900685521,5.00000,1292.00,GBX,100.0,GBP,64.92,GBP,0.32,GBP,64.92


,Action,Time,ISIN,Ticker,Name,Notes,ID,No. of shares,Price / share,Currency (Price / share),Exchange rate,Currency (Result),Total,Currency (Total),Stamp duty reserve tax,Currency (Stamp duty reserve tax),Capital_invested
6,Limit buy,2026-01-15 08:01:36,GB00B2QPKJ12,FRES,Fresnillo,NaN,EOF44912169636,1.0,3646.0,GBX,100.0,GBP,36.64,GBP,0.18,GBP,36.64


Weighted avg function test

In [ ]:
from src.utils.weighted_avg import weighted_avg_v2

try:
    a, b, c = weighted_avg_v2(df_portfolio, df_input_share)
    print(a)
    print(type(a))
    print(b)
    print(type(b))
    print(c)
    print(type(c))
except Exception as e:
    print(f"Error: {type(e).__name__} {e}")

for ticker in input__test_df['Ticker']:
    input_temp = input__test_df[input__test_df['Ticker'] == ticker].copy()
    if ticker in portfolio_test_df['Ticker'].to_list():
        print(f"{ticker} found in portfolio_test_df")
        weighted_price, weighted_time, total_shares, updated_ID_set = weighted_avg_v2(df_portfolio, df_input_share)


display(portfolio_test_df)
display(input__test_df)


3521.513851686704
<class 'numpy.float64'>
2026-01-07 22:10:39.286588416
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
{'EOF44399974076', 'EOF44912169636'}
<class 'set'>


# Set up variables

In [29]:
base_currency = "GBP"

start_date = pd.Timestamp(2025,1,1)
end_date = pd.Timestamp.today().normalize()+pd.Timedelta(days=1)
ROI_target = 0.135

email_recipients = ["ingcarldan@gmail.com"]
email_user = os.getenv("EMAIL_USER")
email_sender_psw = os.getenv("EMAIL_SENDER_PSW")
email_sender = os.getenv("EMAIL_SENDER")

# portfolio setup
new_shares = False
old_portfolio_setup = False
pur = pd.read_csv(portfolio_path, parse_dates=['Purchase_Date'])
pur = pur.drop(columns=['Unnamed: 0'], errors='ignore')

if new_shares == True:
    new_row = {
        "Action": "RR", 
        "Purchase_Date": pd.Timestamp(2026,1,14),#pd.Timestamp.today().normalize(),
        "Purchase_Price": 12.92,
        "Currency": "GBP",
        "Target_ROI":0.135
        }
    pur = pd.concat([pur, pd.DataFrame([new_row])], ignore_index=True)
    if old_portfolio_setup == True:
        purchase_dates = {
            'FRES': pd.Timestamp(2026,1,3),
            'NWG':pd.Timestamp(2026,1,7),
            'GLEN':pd.Timestamp(2026,1,6),
            'RIO':pd.Timestamp(2026,1,8),
        }
        pur = pd.DataFrame(purchase_dates.items(), columns=['Action', 'Purchase_Date'])
        pur[['Purchase_price', 'Current_price' , 'Currency', 'Current_ROI', 'Target_ROI', 'ROI_reached']] = None
        pur['Currency'] = pur['Currency'].astype('string')
        pur['ROI_reached'] = pur['ROI_reached'].astype('bool')
        pur[['Current_ROI', 'Target_ROI']] = pur[['Current_ROI', 'Target_ROI']].astype('float')

        # pp(pur)
        input_cols = ['Purchase_price', 'Currency', 'Target_ROI']
        pur.loc[0, input_cols] = [34.78, 'GBP', 0.135] # Purchase_price FRES
        pur.loc[1, input_cols] = [6.35, 'GBP', 0.135] # Purchase_price NWG
        pur.loc[2, input_cols] = [4.2495, 'GBP', 0.135] # Purchase_price GLEN
        pur.loc[3, input_cols] = [62, 'GBP', 0.135] # Purchase_price RIO
        pur.loc[:, 'ROI_reached'] = False # set default to False



# Portfolio update testing

In [ ]:

print("pur:\n",pur.dtypes)
df_input_share = df_input_share.rename(columns={"Time": "Purchase_Date"})
df_portfolio = df_portfolio.rename(columns={"Time": "Purchase_Date"})
print("df_input_share:\n",df_input_share.dtypes)
print("df_portfolio:\n",df_portfolio.dtypes)

pur:
 Action                    object
Purchase_Date     datetime64[ns]
Purchase_price           float64
Current_price            float64
Currency                  object
Current_ROI              float64
Target_ROI               float64
ROI_reached                 bool
Purchase_Price           float64
dtype: object
df_input_share:
 Action                                       object
Time                                 datetime64[ns]
ISIN                                         object
Ticker                                       object
Name                                         object
Notes                                        object
ID                                           object
No. of shares                               float64
Price / share                               float64
Currency (Price / share)                     object
Exchange rate                               float64
Currency (Result)                            object
Total                                     

# Get TOP 100 shares from FTSE

In [ ]:
ftse100 = get_ftse100()
ftse100["Yahoo_Ticker"] = ftse100["Ticker"] + ".L"
shares_lse = ftse100["Yahoo_Ticker"].to_list()

# SHARES PRICES WITH INFO

In [ ]:
df_shares_fund, failed_tickers_list = get_share_prices_2_with_fundamentals(
    tickers=shares_lse,# list with items ending with ".L"
    start=start_date,
    end=end_date,
    base_currency = base_currency,
    vol_window = 20,
    
)

actions_list   = df_shares_fund.columns.get_level_values("ACTION").unique().to_list()
currencies_list = df_shares_fund.columns.get_level_values("CURRENCY").unique()
metrics   = df_shares_fund.columns.get_level_values("METRIC").unique()
display(df_shares_fund)

# GET PURCHASE PRICES IF NOT PROVIDED

In [ ]:
# Get purchase price if not provided using timestamp as reference
# if timestamp not found in df, revert to previous available date

for action, purchase_date in pur[['Action', 'Purchase_Date']].itertuples(index=False):

    idx = pur.index[
        (pur['Action'] == action) &
        (pur['Purchase_Date'] == purchase_date)
    ][0]
    action_full = next(act for act in actions_list if act.startswith(action))
    currency = action_full[-3:]

    # print(f"Idx for action {action} and purchase date {purchase_date} is {idx}")

    if pd.isna(pur.loc[idx, 'Purchase_price']):
       
        HIGH = get_purchase_price(
            df=df_shares_fund,
            action=action_full,
            currency=currency,
            metric="HIGH",
            date=purchase_date          # <-- scalar Timestamp
        )
        LOW = get_purchase_price(
            df=df_shares_fund,
            action=action_full,
            currency=currency,
            metric="LOW",
            date=purchase_date          # <-- scalar Timestamp
        )
        pur.loc[idx, 'Purchase_price'] = (HIGH + LOW) / 2

    current_value = df_shares_fund[(action_full,action_full[-3:],'CLOSE')].iloc[-1]
    pur.loc[idx, 'Current_price'] = current_value
    pur.loc[idx, 'Currency'] = currency
    pur.loc[idx, 'Current_ROI'] = round((current_value - pur.loc[idx, 'Purchase_price']) / pur.loc[idx, 'Purchase_price'],4)
    pur.loc[idx, 'ROI_reached'] = pur.loc[idx, 'Current_ROI'] >= pur.loc[idx, 'Target_ROI']

# Save updated portfolio
pur.to_csv(portfolio_path, index=False)

In [ ]:
display(pur)

# EXTRACT UNDERVALUED SHARES

In [ ]:
df_fund = extract_latest_fundamentals(
    df=df_shares_fund,
    evaluation_date=end_date,
)

undervalued_shares = detect_undervalued(df_fund)
filt = undervalued_shares[undervalued_shares["UndervaluedScore"] > 0]
undervalued_shares_list = filt.index.to_list()
filt["Ticker"] = filt.index.str.split(".").str[0]

filt["Company"] = (
    filt["Ticker"]
    .map(ftse100.set_index("Ticker")["Company"])
)
currency_convertion = [s.split('_')[1] for s in filt.index.to_list()]

filt['original currency'] = [s.split('→')[0] for s in currency_convertion]
filt['converted currency'] = [s.split('→')[1] for s in currency_convertion]
filt = filt[['Company', 'Ticker', 'UndervaluedScore', 'original currency', 'converted currency', 
                'Price', 'EPS', 'BookValue', 'Dividend', 'P/E', 'P/B']]
ROI( # from src.plot_shares_ROI import plot_candles_volatility_volume_roi
    df=df_shares_fund,
    actions=filt.index.to_list(), # list with items ending with ".L_GBp→GBP"
    start=df_shares_fund.index.min(),
    end=df_shares_fund.index.max(),
    roi_target=ROI_target,
    plot_purchase=False
)

# SEND EMAIL FOR UNDERVALUED SHARES

In [ ]:
try:
    text_body, html_body, inline_images = build_undervalued_shares_email_with_images(
        df_undervalued_shares=filt,
        image_dir='output'
    )

except Exception as e:
    print(f"ERROR in build_undervalued_shares_email_with_images {type(e).__name__}: {e}")
try:
    send_email_html_multi_inline_images(
        smtp_server="smtp.gmail.com",
        smtp_port=587,
        username=email_user,
        password=email_sender_psw,
        sender=email_sender,
        recipients=email_recipients,
        subject=f"FTSE100 - Undevalued shares – {datetime.today():%d-%m-%Y %H:%M}",
        text_body=text_body,
        html_body=html_body,
        inline_images=inline_images,
    )
except Exception as e:
    print(f"Could not run send_email_html_multi_inline_images {type(e).__name__}: {e}")
        


portfolio = {}
SHARES_FULL_LIST = [s + '.L_GBp→GBP' if not s.endswith('.L_GBp→GBP') else s for s in list(purchase_dates.keys())]

for action in SHARES_FULL_LIST:
    try:
        action_clean = action.split(".L")[0]
        portfolio[action_clean] = get_first_roi_hit(
        df=df_shares_fund,
        action=action,
        purchase_dates=purchase_dates,
        roi_target=ROI_target
    )
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

#  PLOT SHARE - plot_candles_volatility_volume_roi 


In [ ]:
filtered_actions = [s for s in actions_list if s.startswith(tuple(pur['Action'].to_list()))]

# save input for DEBUG
debug_file_option = False
if debug_file_option == True:
    import pickle
    df_shares_fund.to_pickle("df_shares_fund.pkl")
    pur.to_pickle("pur.pkl")

    with open("filtered_actions.pkl", "wb") as f:
        pickle.dump(filtered_actions, f)

    with open("ROI_target.pkl", "wb") as f:
        pickle.dump(ROI_target, f)



ROI( # from src.plot_shares_ROI import plot_candles_volatility_volume_roi
    df=df_shares_fund,
    actions=filtered_actions,
    start=df_shares_fund.index.min(),
    end=df_shares_fund.index.max(),
    purchase_dates=pur,
    roi_target=ROI_target,
)

# ROI email HTML body setup

In [ ]:
image_dir = os.path.join(os.getcwd(), "output")
try:
    roi_text_body, roi_html_body, roi_inline_images = build_roi_email_from_df(
        df=pur,
        action_list = actions_list,
        image_dir = image_dir,
    )
    # print(f"actions_list:\n {actions_list}")
    # print(f"text_body: {text_body}\nhtml_body: {html_body}\ninline_images keys: {list(inline_images.keys())}\n")
except Exception as e:
    print(f"Could not run build_roi_email_from_df {type(e).__name__}: {e}")


DEBUG CELL

print("roi_text_body")
pp(roi_text_body)
print("roi_html_body")
pp(roi_html_body)
print("roi_inline_images")
pp(roi_inline_images)

# Send email (TLS via your utility)

In [ ]:

try:
    send_email_html_multi_inline_images(
        smtp_server="smtp.gmail.com",
        smtp_port=587,
        username=email_user,
        password=email_sender_psw,
        sender=email_sender,
        recipients=email_recipients,
        subject=f"FTSE100 - ROI targets – {datetime.today():%d-%m-%Y %H:%M}",
        text_body=roi_text_body,
        html_body=roi_html_body,
        inline_images=roi_inline_images,
    )
except Exception as e:
    print(f"Could not run send_email_html_multi_inline_images {type(e).__name__}: {e}")